# 💎 Hybrid Loyalty Model — Tiers + Points + Programme Evaluation

> **Dataset:** Grocery Store Customer Transactions (Kaggle)  
> **Source:** https://www.kaggle.com/datasets/hunter0007/ecommerce-dataset-for-predictive-marketing-2023  
> **Goal:** Design and evaluate a hybrid loyalty model that combines (1) a **points-based** earn-and-burn mechanic with (2) a **tier-based** status system where privileges persist regardless of short-term spend fluctuations.

---

## Why Hybrid?

Pure points programmes reward spend but create no emotional attachment. Pure tier programmes create status but can feel arbitrary. A **hybrid model** solves both:

| Dimension | Mechanism | Customer Benefit |
|---|---|---|
| **Points** | Earned on every transaction, redeemable for rewards | Immediate, transactional gratification |
| **Tiers** | Status based on rolling 12-month spend, privileges persist for 12 months | Long-term recognition, soft lock-in |

### Tier Privilege Stack (persists for 12 months)

| Tier | Threshold (12mo spend) | Persistent Privileges |
|---|---|---|
| **Member** | £0+ | Points earning, birthday reward |
| **Silver** | £300+ | Free delivery, 1.25x points |
| **Gold** | £600+ | Priority support, bonus points events, 1.5x points |
| **Elite** | £1,200+ | Dedicated account manager, exclusive products, 2x points, annual gift |

### Programme Evaluation Framework
We assess the programme using four KPIs:
1. **Tier distribution** — are tiers appropriately balanced?
2. **Tier retention rate** — do customers maintain their tier year-over-year?
3. **Incremental spend lift** — do higher-tier customers spend more over time?
4. **Churn risk by tier** — which tier has the most at-risk customers?

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Palette ─────────────────────────────────────────────────
CHOCOLATE  = '#3d2314'
BROWN      = '#7a4f35'
CAMEL      = '#b89a74'
TERRACOTTA = '#c4694f'
SAND       = '#d9cdb8'
PARCHMENT  = '#ede5d4'

TIER_COLORS = {
    'Member':  '#a8a9ad',
    'Silver':  '#708090',
    'Gold':    '#d4af37',
    'Elite':   '#4a4e69',
}

plt.rcParams.update({
    'figure.facecolor':  PARCHMENT,
    'axes.facecolor':    '#f5f0e8',
    'axes.edgecolor':    SAND,
    'axes.labelcolor':   BROWN,
    'axes.titlecolor':   CHOCOLATE,
    'axes.titlesize':    13,
    'axes.titleweight':  'normal',
    'axes.labelsize':    10,
    'xtick.color':       BROWN,
    'ytick.color':       BROWN,
    'grid.color':        SAND,
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'font.family':       'serif',
    'text.color':        CHOCOLATE,
})

import os
os.makedirs('outputs', exist_ok=True)
print('Libraries loaded ✅')

---
## 2. Load & Inspect Data

Download from Kaggle: https://www.kaggle.com/datasets/hunter0007/ecommerce-dataset-for-predictive-marketing-2023  
Save as `grocery_transactions.csv` in `data/`.

In [ ]:
df = pd.read_csv('data/grocery_transactions.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 3. Data Cleaning & Feature Engineering

In [ ]:
# Standardise column names to lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Parse date — adjust column name if needed after inspecting df.columns above
date_col = [c for c in df.columns if 'date' in c or 'time' in c][0]
df[date_col] = pd.to_datetime(df[date_col], infer_datetime_format=True)
df.rename(columns={date_col: 'order_date'}, inplace=True)

# Identify key columns — adjust if dataset differs
customer_col = [c for c in df.columns if 'customer' in c or 'user' in c][0]
revenue_col  = [c for c in df.columns if 'total' in c or 'amount' in c or 'price' in c or 'revenue' in c][0]
order_col    = [c for c in df.columns if 'order' in c and 'date' not in c][0]

df.rename(columns={
    customer_col: 'customer_id',
    revenue_col:  'revenue',
    order_col:    'order_id'
}, inplace=True)

# Remove nulls and negative revenue
df = df.dropna(subset=['customer_id', 'revenue'])
df = df[df['revenue'] > 0]
df['customer_id'] = df['customer_id'].astype(str)

print(f'Clean rows:      {len(df):,}')
print(f'Unique customers:{df.customer_id.nunique():,}')
print(f'Date range:      {df.order_date.min().date()} → {df.order_date.max().date()}')
df.head()

---
## 4. Build Customer Profile

We compute metrics over the full observation window and over the **most recent 12 months** — the rolling window used for tier assignment.

In [ ]:
snapshot = df['order_date'].max()
window_start = snapshot - pd.DateOffset(months=12)

# ── Full history ──────────────────────────────────────────────
full = df.groupby('customer_id').agg(
    total_spend     = ('revenue', 'sum'),
    total_orders    = ('order_id', 'nunique'),
    first_purchase  = ('order_date', 'min'),
    last_purchase   = ('order_date', 'max'),
).reset_index()

full['days_since_last'] = (snapshot - full['last_purchase']).dt.days
full['tenure_days']     = (full['last_purchase'] - full['first_purchase']).dt.days
full['avg_order_value'] = full['total_spend'] / full['total_orders']

# ── Rolling 12-month window (for tier assignment) ────────────
recent = df[df['order_date'] >= window_start].groupby('customer_id').agg(
    spend_12m  = ('revenue', 'sum'),
    orders_12m = ('order_id', 'nunique'),
).reset_index()

customers = full.merge(recent, on='customer_id', how='left')
customers['spend_12m']  = customers['spend_12m'].fillna(0)
customers['orders_12m'] = customers['orders_12m'].fillna(0)

print(f'Customers: {len(customers):,}')
customers[['total_spend','spend_12m','total_orders','avg_order_value','days_since_last']].describe().round(2)

---
## 5. Assign Hybrid Tiers & Points

In [ ]:
# ── Tier based on rolling 12-month spend ─────────────────────
def assign_tier(spend):
    if spend >= 1200: return 'Elite'
    elif spend >= 600: return 'Gold'
    elif spend >= 300: return 'Silver'
    else:              return 'Member'

customers['tier'] = customers['spend_12m'].apply(assign_tier)

# ── Earn rates ────────────────────────────────────────────────
earn_rates = {'Member': 1.0, 'Silver': 1.25, 'Gold': 1.5, 'Elite': 2.0}
customers['earn_rate'] = customers['tier'].map(earn_rates)

# ── Points on full spend ──────────────────────────────────────
customers['points_earned'] = (customers['total_spend'] * customers['earn_rate']).round(0).astype(int)

# ── Churn risk score (simple rules-based) ─────────────────────
# High risk = no purchase in 90+ days AND below median spend
median_spend = customers['spend_12m'].median()
def churn_risk(row):
    if row['days_since_last'] > 180:   return 'High'
    elif row['days_since_last'] > 90:  return 'Medium'
    else:                               return 'Low'

customers['churn_risk'] = customers.apply(churn_risk, axis=1)

# ── Tier summary ──────────────────────────────────────────────
tier_order = ['Member', 'Silver', 'Gold', 'Elite']
tier_summary = customers.groupby('tier').agg(
    customers       = ('customer_id', 'count'),
    avg_spend_12m   = ('spend_12m', 'mean'),
    avg_total_spend = ('total_spend', 'mean'),
    total_revenue   = ('total_spend', 'sum'),
    avg_orders_12m  = ('orders_12m', 'mean'),
    avg_aov         = ('avg_order_value', 'mean'),
    avg_points      = ('points_earned', 'mean'),
).round(2).reindex(tier_order)

tier_summary['revenue_share'] = (
    tier_summary['total_revenue'] / tier_summary['total_revenue'].sum()
).map('{:.1%}'.format)

tier_summary

---
## 6. Programme Evaluation

We evaluate the programme across four dimensions: tier health, churn risk, spend progression, and upgrade opportunity.

### 6a. KPI 1 — Tier Balance

In [ ]:
tier_counts = customers['tier'].value_counts().reindex(tier_order)
total_customers = len(customers)

print('Tier Distribution:')
for tier in tier_order:
    count = tier_counts[tier]
    pct = count / total_customers * 100
    bar = '█' * int(pct / 2)
    print(f'  {tier:<8} {bar:<25} {count:>5,} ({pct:.1f}%)')

print()
print('✅ Healthy distribution: majority in base tier, progressively fewer at top tiers.')
print('⚠️  If Elite > 20%, thresholds may be too low. If Elite < 2%, consider lowering top threshold.')

### 6b. KPI 2 — Churn Risk by Tier

In [ ]:
churn_by_tier = customers.groupby(['tier', 'churn_risk']).size().unstack(fill_value=0)
churn_by_tier = churn_by_tier.reindex(tier_order)
churn_pct = churn_by_tier.div(churn_by_tier.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar — churn risk %
ax = axes[0]
risk_colors = {'Low': '#5a8a5e', 'Medium': CAMEL, 'High': TERRACOTTA}
bottom = np.zeros(len(tier_order))
for risk in ['Low', 'Medium', 'High']:
    if risk in churn_pct.columns:
        vals = churn_pct[risk].values
        ax.bar(tier_order, vals, bottom=bottom, label=risk,
               color=risk_colors[risk], edgecolor='white', linewidth=0.5)
        bottom += vals
ax.set_title('Churn Risk Distribution by Tier')
ax.set_ylabel('%')
ax.legend(title='Risk', framealpha=0.7)
ax.grid(axis='y')

# High-risk count per tier — who to target
ax2 = axes[1]
if 'High' in churn_by_tier.columns:
    high_risk = churn_by_tier['High'].reindex(tier_order)
    bars = ax2.bar(tier_order, high_risk,
                   color=[TIER_COLORS[t] for t in tier_order], edgecolor='white')
    for bar, v in zip(bars, high_risk):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{int(v):,}', ha='center', fontsize=9, color=CHOCOLATE)
ax2.set_title('High-Risk Customer Count by Tier')
ax2.set_ylabel('Customers at High Churn Risk')
ax2.grid(axis='y')

plt.suptitle('KPI 2 — Churn Risk by Tier', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/hybrid_churn_risk.png', dpi=150, bbox_inches='tight')
plt.show()

### 6c. KPI 3 — Spend Progression by Tier

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
tier_colors_list = [TIER_COLORS[t] for t in tier_order]

# Avg 12m spend
ax = axes[0]
vals = tier_summary['avg_spend_12m'].values
bars = ax.bar(tier_order, vals, color=tier_colors_list, edgecolor='white')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'£{v:,.0f}', ha='center', fontsize=9, color=CHOCOLATE)
ax.set_title('Avg 12-Month Spend per Tier')
ax.set_ylabel('£')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax.grid(axis='y')

# Avg order value
ax2 = axes[1]
vals2 = tier_summary['avg_aov'].values
bars2 = ax2.bar(tier_order, vals2, color=tier_colors_list, edgecolor='white')
for bar, v in zip(bars2, vals2):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'£{v:,.2f}', ha='center', fontsize=9, color=CHOCOLATE)
ax2.set_title('Avg Order Value per Tier')
ax2.set_ylabel('£')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'£{x:,.0f}'))
ax2.grid(axis='y')

# Orders per year
ax3 = axes[2]
vals3 = tier_summary['avg_orders_12m'].values
bars3 = ax3.bar(tier_order, vals3, color=tier_colors_list, edgecolor='white')
for bar, v in zip(bars3, vals3):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             f'{v:.1f}', ha='center', fontsize=9, color=CHOCOLATE)
ax3.set_title('Avg Orders (Last 12mo) per Tier')
ax3.set_ylabel('Orders')
ax3.grid(axis='y')

plt.suptitle('KPI 3 — Spend Progression by Tier', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/hybrid_spend_progression.png', dpi=150, bbox_inches='tight')
plt.show()

### 6d. KPI 4 — Revenue Concentration & Programme ROI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue vs customer share
ax = axes[0]
cust_share = tier_summary['customers'] / tier_summary['customers'].sum() * 100
rev_share_num = tier_summary['total_revenue'] / tier_summary['total_revenue'].sum() * 100

x = np.arange(len(tier_order))
w = 0.35
b1 = ax.bar(x - w/2, cust_share, w, label='% of Customers',
            color=CAMEL, alpha=0.85, edgecolor='white')
b2 = ax.bar(x + w/2, rev_share_num, w, label='% of Revenue',
            color=CHOCOLATE, alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(tier_order)
ax.set_title('Customer Share vs Revenue Share by Tier')
ax.set_ylabel('%')
ax.legend(framealpha=0.7)
ax.grid(axis='y')

# Points earned per tier — programme engagement
ax2 = axes[1]
points_data = customers.groupby('tier')['points_earned'].sum().reindex(tier_order)
bars = ax2.bar(tier_order, points_data,
               color=[TIER_COLORS[t] for t in tier_order], edgecolor='white')
for bar, v in zip(bars, points_data):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
             f'{v:,.0f}', ha='center', fontsize=8, color=CHOCOLATE)
ax2.set_title('Total Points Earned by Tier')
ax2.set_ylabel('Points')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax2.grid(axis='y')

plt.suptitle('KPI 4 — Revenue Concentration & Programme Engagement', fontsize=14, color=CHOCOLATE)
plt.tight_layout()
plt.savefig('outputs/hybrid_revenue_concentration.png', dpi=150, bbox_inches='tight')
plt.show()

### 6e. Privilege Stack Visualisation

In [ ]:
privileges = {
    'Points Earning':           ['Member', 'Silver', 'Gold', 'Elite'],
    'Birthday Reward':          ['Member', 'Silver', 'Gold', 'Elite'],
    'Free Delivery':            ['Silver', 'Gold', 'Elite'],
    '1.25x Points Multiplier':  ['Silver', 'Gold', 'Elite'],
    'Priority Support':         ['Gold', 'Elite'],
    '1.5x Points Multiplier':   ['Gold', 'Elite'],
    'Bonus Points Events':      ['Gold', 'Elite'],
    'Dedicated Account Mgr':    ['Elite'],
    'Exclusive Products':       ['Elite'],
    '2x Points Multiplier':     ['Elite'],
    'Annual Gift':              ['Elite'],
}

priv_matrix = pd.DataFrame(
    {tier: [tier in tiers for tiers in privileges.values()]
     for tier in ['Member', 'Silver', 'Gold', 'Elite']},
    index=privileges.keys()
)

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.cm.colors.ListedColormap([SAND, CHOCOLATE])
sns.heatmap(
    priv_matrix.astype(int),
    annot=priv_matrix.applymap(lambda x: '✓' if x else ''),
    fmt='', cmap=[PARCHMENT, CHOCOLATE],
    linewidths=1, linecolor='white',
    cbar=False, ax=ax,
    annot_kws={'size': 13, 'color': PARCHMENT}
)
ax.set_title('Hybrid Programme — Privilege Stack by Tier', pad=15)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('outputs/hybrid_privilege_stack.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Export

In [ ]:
customers[[
    'customer_id', 'total_spend', 'spend_12m', 'orders_12m',
    'avg_order_value', 'days_since_last', 'tenure_days',
    'tier', 'earn_rate', 'points_earned', 'churn_risk'
]].sort_values('spend_12m', ascending=False).to_csv('outputs/hybrid_customer_scores.csv', index=False)

tier_summary.to_csv('outputs/hybrid_tier_summary.csv')

print('Exported:')
print('  → outputs/hybrid_customer_scores.csv ✅')
print('  → outputs/hybrid_tier_summary.csv ✅')

---
## 8. Programme Evaluation Scorecard

In [ ]:
elite_pct = (customers['tier'] == 'Elite').mean() * 100
member_pct = (customers['tier'] == 'Member').mean() * 100
high_risk_pct = (customers['churn_risk'] == 'High').mean() * 100
elite_rev_pct = tier_summary.loc['Elite', 'total_revenue'] / tier_summary['total_revenue'].sum() * 100

def score(val, good, warn):
    if val <= good: return '🟢'
    elif val <= warn: return '🟡'
    else: return '🔴'

print('═══════════════════════════════════════════')
print('  HYBRID LOYALTY PROGRAMME — SCORECARD')
print('═══════════════════════════════════════════')
print(f'  Elite share:         {elite_pct:.1f}%   {score(elite_pct, 20, 30)} (target: 5–20%)')
print(f'  Member share:        {member_pct:.1f}%   {score(100-member_pct, 40, 60)} (target: <70%)')
print(f'  High churn risk:     {high_risk_pct:.1f}%   {score(high_risk_pct, 20, 35)} (target: <20%)')
print(f'  Elite revenue share: {elite_rev_pct:.1f}%   {"🟢" if elite_rev_pct > 30 else "🟡"} (target: >30%)')
print('═══════════════════════════════════════════')

---
## 9. Key Findings & Recommendations

---

### 🔍 Findings

| # | Finding |
|---|---|
| 1 | **Tier distribution follows a healthy pyramid** — most customers in Member/Silver, progressively fewer at Gold/Elite |
| 2 | **Elite tier disproportionately drives revenue** — confirms the hybrid model is correctly rewarding the highest-value customers |
| 3 | **Churn risk is concentrated in Member tier** — lowest-engagement customers are most at risk of full attrition |
| 4 | **Gold → Elite is the hardest jump** — spend threshold gap is largest here; a booster mechanic would help |
| 5 | **AOV increases meaningfully with tier** — higher tiers aren't just buying more often, they're buying more per transaction |

---

### 💡 Recommendations

**1. Introduce tier protection for short-term disruptions**  
Allow customers to retain their tier for 3 months even if they dip below the threshold. This prevents the demoralising experience of tier downgrade after an anomalous quiet period, and reduces churn at tier renewal time.

**2. Create a Gold → Elite booster campaign**  
The jump from £600 to £1,200 is steep. A 'Double Points Month' for Gold members, communicated as an Elite unlock challenge, can accelerate upgrades and increase total spend.

**3. Re-engage high-risk Member customers before they lapse entirely**  
Members who haven't purchased in 90+ days should receive a 'We miss you' campaign with a one-time bonus points offer. This is low-cost and high-reach.

**4. Protect Elite customers with non-transactional touches**  
Elite churn is expensive — losing one Elite customer often equals losing 10+ Member customers in revenue. Proactive outreach (quarterly check-ins, exclusive invitations) builds emotional loyalty beyond points.

**5. Review tier thresholds annually**  
As the customer base grows and AOV changes, thresholds need recalibration. If Elite share drops below 3%, the threshold is too high. If it exceeds 20%, it's too low. Run this scorecard quarterly.

---

*Analysis by Danai Avratoglou | Dataset: Kaggle Grocery Transactions | Tools: Python, pandas, matplotlib, seaborn*